# Notebook 04 — Model Evaluation

Complete evaluation of the BiLSTM classifier and XGBoost form scorer.

**What we evaluate:**
1. Classifier: accuracy, macro F1, per-class precision/recall
2. Confusion matrix (counts + normalized)
3. Per-class ROC curves + AUC scores
4. Attention weight analysis — what gait phase triggers each fault?
5. Form scorer: MAE, R², predicted vs actual scatter
6. Error analysis — what does the model get wrong?

In [ ]:
import sys
sys.path.insert(0, '..')

import json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, f1_score,
    roc_auc_score, roc_curve,
    mean_absolute_error, r2_score
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize

from src.modeling.bilstm_model import RunningFormClassifier
from src.modeling.form_scorer import SCORER_FEATURES, aggregate_clip_features, build_score_labels

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

CLASS_ORDER  = ['good_form', 'overstriding', 'forward_lean', 'arm_crossing']
CLASS_COLORS = {'good_form':'#2ecc71','overstriding':'#e74c3c',
                'forward_lean':'#f39c12','arm_crossing':'#9b59b6'}

SEQ_DIR   = Path('../data/processed/sequences')
CKPT_PATH = Path('../models/classifier/best_model.pt')
SCOR_PATH = Path('../models/form_scorer.pkl')
FEAT_PATH = Path('../data/processed/features/biomech_features.csv')
print('✅ Libraries loaded')

## 1. Load Saved Evaluation Results

In [ ]:
eval_path = Path('../docs/evaluation/evaluation_results.json')
if eval_path.exists():
    with open(eval_path) as f:
        results = json.load(f)
    print('SAVED EVALUATION RESULTS')
    print('='*45)
    for k, v in results.items():
        print(f'  {k:30s}: {v}')
else:
    print('No saved results. Run: python src/modeling/evaluate.py')
    results = {}

## 2. BiLSTM Classifier Evaluation

In [ ]:
if CKPT_PATH.exists() and (SEQ_DIR / 'X_test.npy').exists():
    device = torch.device('cpu')
    ckpt   = torch.load(CKPT_PATH, map_location=device)
    model  = RunningFormClassifier(
        input_size=ckpt['input_size'], hidden_size=ckpt['hidden_size'],
        num_layers=ckpt['num_layers'], num_classes=ckpt['num_classes'],
        dropout=ckpt['dropout'],
    ).to(device)
    model.load_state_dict(ckpt['model_state'])
    model.eval()

    with open(SEQ_DIR / 'label_encoder.pkl', 'rb') as f:
        le = pickle.load(f)

    X_test = torch.FloatTensor(np.load(SEQ_DIR / 'X_test.npy'))
    y_test = np.load(SEQ_DIR / 'y_test.npy')

    with torch.no_grad():
        logits, attn = model(X_test, return_attention=True)
    probs = F.softmax(logits, dim=1).numpy()
    preds = probs.argmax(1)

    acc = accuracy_score(y_test, preds)
    f1  = f1_score(y_test, preds, average='macro', zero_division=0)
    print(f'Accuracy : {acc:.4f}')
    print(f'F1 macro : {f1:.4f}')
    print()
    print(classification_report(y_test, preds, target_names=le.classes_))
else:
    print('Train the model first: python src/modeling/train_classifier.py')

## 3. Confusion Matrix + ROC Curves

In [ ]:
if 'probs' in dir():
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    classes = le.classes_
    
    # Confusion matrix — counts
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes, ax=axes[0])
    axes[0].set_title('Confusion Matrix (counts)', fontweight='bold')
    axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')
    
    # Confusion matrix — normalized
    cm_norm = cm.astype(float) / cm.sum(1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=classes, yticklabels=classes, ax=axes[1])
    axes[1].set_title('Confusion Matrix (normalized)', fontweight='bold')
    axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')
    
    # ROC curves
    y_bin = label_binarize(y_test, classes=list(range(len(classes))))
    for i, (cls, color) in enumerate(CLASS_COLORS.items()):
        fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
        auc_v = roc_auc_score(y_bin[:, i], probs[:, i])
        axes[2].plot(fpr, tpr, color=color, lw=2, label=f'{cls} (AUC={auc_v:.2f})')
    axes[2].plot([0,1],[0,1],'k--',lw=1,alpha=0.4)
    axes[2].set_xlabel('FPR'); axes[2].set_ylabel('TPR')
    axes[2].set_title('One-vs-Rest ROC Curves', fontweight='bold')
    axes[2].legend(fontsize=8)
    
    plt.suptitle('BiLSTM Classifier — Test Set Evaluation', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 4. Attention Analysis — Which Gait Phase Matters?

In [ ]:
if 'attn' in dir() and 'y_test' in dir():
    attn_np = attn.detach().numpy()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    x = np.linspace(0, 1, attn_np.shape[1])
    
    # Mean attention per class
    for i, (cls, color) in enumerate(CLASS_COLORS.items()):
        mask = y_test == i
        if mask.sum() > 0:
            mean_a = attn_np[mask].mean(0)
            std_a  = attn_np[mask].std(0)
            axes[0].plot(x, mean_a, color=color, lw=2, label=cls)
            axes[0].fill_between(x, mean_a-std_a, mean_a+std_a, color=color, alpha=0.1)
    axes[0].axvline(0.5, color='gray', linestyle=':', label='Mid-swing')
    axes[0].set_xlabel('Gait phase (normalized)'); axes[0].set_ylabel('Mean attention weight')
    axes[0].set_title('Attention by Form Class'); axes[0].legend(fontsize=8)
    
    # Attention heatmap for first 30 samples
    n_show = min(30, len(attn_np))
    row_labels = [le.classes_[y_test[i]] for i in range(n_show)]
    im = axes[1].imshow(attn_np[:n_show], aspect='auto', cmap='YlOrRd')
    axes[1].set_xlabel('Frame'); axes[1].set_ylabel('Sample')
    axes[1].set_title(f'Attention Heatmap (first {n_show} test samples)')
    plt.colorbar(im, ax=axes[1])
    
    plt.suptitle('Attention Weight Analysis', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Which phase has highest attention?
    mean_all = attn_np.mean(0)
    peak_frame = int(mean_all.argmax())
    print(f'Highest mean attention: frame {peak_frame}/{len(mean_all)} ({peak_frame/len(mean_all):.0%} of gait cycle)')
    print('Interpretation: model focuses most on this gait phase for classification')

## 5. Form Scorer Evaluation

In [ ]:
if SCOR_PATH.exists() and FEAT_PATH.exists():
    with open(SCOR_PATH, 'rb') as f:
        artifact = pickle.load(f)
    pipeline   = artifact['pipeline']
    feat_cols  = artifact['feature_cols']
    
    df       = pd.read_csv(FEAT_PATH)
    clip_df  = aggregate_clip_features(df)
    avail    = [c for c in feat_cols if c in clip_df.columns]
    X        = clip_df[avail].fillna(0)
    y        = build_score_labels(clip_df)
    _, X_te, _, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    y_pred   = pipeline.predict(X_te)
    
    mae = mean_absolute_error(y_te, y_pred)
    r2  = r2_score(y_te, y_pred)
    print(f'Form Scorer — MAE: {mae:.2f} pts | R²: {r2:.4f}')
    
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    
    # Scatter
    axes[0].scatter(y_te, y_pred, alpha=0.5, color='#4C72B0', s=30)
    lims = [min(y_te.min(), y_pred.min())-2, max(y_te.max(), y_pred.max())+2]
    axes[0].plot(lims, lims, 'r--', lw=1.5, label='Perfect')
    axes[0].set_xlabel('True Form Score'); axes[0].set_ylabel('Predicted')
    axes[0].set_title(f'Predicted vs Actual (MAE={mae:.1f})', fontweight='bold')
    axes[0].legend()
    
    # Feature importance
    reg = pipeline.named_steps['reg']
    imp = reg.feature_importances_
    idx = np.argsort(imp)[::-1][:10]
    axes[1].barh([avail[i] for i in idx], imp[idx], color='#55A868', edgecolor='white')
    axes[1].invert_yaxis()
    axes[1].set_title('Top Feature Importances', fontweight='bold')
    
    plt.suptitle('Form Scorer Evaluation', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Train the form scorer first: python src/modeling/form_scorer.py')

## 6. Error Analysis — What Does the Model Get Wrong?

In [ ]:
if 'probs' in dir() and 'y_test' in dir():
    correct = preds == y_test
    print(f'Total test : {len(y_test)}')
    print(f'Correct    : {correct.sum()} ({correct.mean():.1%})')
    print(f'Wrong      : {(~correct).sum()}')
    print()
    
    # Most confused pairs
    errors = [(le.classes_[y_test[i]], le.classes_[preds[i]])
              for i in range(len(y_test)) if preds[i] != y_test[i]]
    from collections import Counter
    error_counts = Counter(errors)
    print('Most common misclassifications:')
    for (true_c, pred_c), count in error_counts.most_common(10):
        print(f'  {true_c:20s} → {pred_c:20s}: {count}')
    
    # Confidence on errors vs correct
    fig, ax = plt.subplots(figsize=(9, 4))
    correct_conf = probs[correct].max(1)
    wrong_conf   = probs[~correct].max(1)
    ax.hist(correct_conf, bins=20, alpha=0.6, color='#2ecc71', label=f'Correct (n={correct.sum()})', edgecolor='white')
    ax.hist(wrong_conf,   bins=20, alpha=0.6, color='#e74c3c', label=f'Wrong (n={(~correct).sum()})',   edgecolor='white')
    ax.axvline(0.5, color='black', linestyle='--', linewidth=1.5, label='0.5 threshold')
    ax.set_xlabel('Predicted Confidence'); ax.set_ylabel('Count')
    ax.set_title('Confidence Distribution: Correct vs Incorrect', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()
    print('\nObservation: Errors cluster near the 0.5 threshold — borderline cases are hardest.')